# NotebookLM Kit — Unified Artifact Pipeline

One pipeline that creates **any** artifact type, one per source, then polls and downloads.

1. **Setup** — authenticate and verify `tsx`
2. **Config** — set your notebook ID
3. **Pick artifact type** — choose `ARTIFACT_TYPE` and the matching customization block
4. **List sources** — preview sources; pick indices to filter
5. **Create** — one call: `create_artifacts(NOTEBOOK_ID, ARTIFACT_TYPE, sources, CUSTOMIZATION, INSTRUCTIONS, creds, dry_run=...)`
6. **Poll** — wait for generation, or resume from a previous run
7. **Download** — save outputs to `outputs/<type>/`

---

### ⚠️ Important: QUIZ ↔ FLASHCARDS ambiguity

Both `QUIZ` and `FLASHCARDS` are sent to NotebookLM's backend as **API type 4** (see `src/services/artifacts.ts` → `getApiTypeNumber`). The discriminator is the **shape of the customization array**:

| Type | Inner array length | `[difficulty, 1]` at index |
|---|---|---|
| `QUIZ`       | 8 elements | 7 |
| `FLASHCARDS` | 7 elements | 6 |

When the SDK parses a response and can't recognise either shape, it **defaults to `QUIZ`** (see `mapApiTypeToArtifactType`). That's why flashcards sometimes show up labelled as quizzes.

**Mitigation built into this notebook:** the flashcard template below always sets `numberOfCards` and `difficulty`, which forces the unambiguous 7-element customization on the wire and in the response.

In [ ]:
import importlib, config, pipeline, pipeline._core, pipeline._sources, pipeline._artifacts, pipeline.config
importlib.reload(config)
importlib.reload(pipeline.config)
importlib.reload(pipeline._core)
importlib.reload(pipeline._sources)
importlib.reload(pipeline._artifacts)
importlib.reload(pipeline)

from pipeline import load_credentials, check_tsx, login, SDK_ROOT
from config import NOTEBOOK_ID, VIDEO_PROMPT, INFOGRAPHIC_PROMPT, SLIDES_PROMPT

# First time only — opens a browser window for Google login, saves credentials.json:
# login()

creds = load_credentials(mode="patchright")
check_tsx()

Credentials ready — token: 42 chars, cookies: 2548 chars
tsx: tsx v4.22.3
node v20.17.0


## Config

Find your notebook ID in the NotebookLM URL:  
`https://notebooklm.google.com/notebook/` **`<YOUR_NOTEBOOK_ID>`**

In [ ]:
# NOTEBOOK_ID = "00000000-0000-0000-0000-000000000000"  # ← paste your notebook ID here

## Pick Artifact Type

All customization shapes below are taken directly from `src/types/artifact.ts`. Set `ARTIFACT_TYPE` to one of:

| Type           | Customization supported | One-per-source useful? |
|----------------|:-----------------------:|:----------------------:|
| `FLASHCARDS`   | ✅ | ✅ |
| `QUIZ`         | ✅ | ✅ |
| `VIDEO`        | ✅ | ✅ |
| `AUDIO`        | ✅ (sources always all) | ❌ (always whole notebook) |
| `SLIDE_DECK`   | ✅ | ✅ |
| `INFOGRAPHIC`  | ✅ | ✅ |
| `REPORT`       | ❌ | ✅ |
| `MIND_MAP`     | ❌ | ✅ |

The dict assigned to `CUSTOMIZATION` **must** match `ARTIFACT_TYPE` — cross-type fields are silently ignored by the SDK and can trigger the QUIZ/FLASHCARDS ambiguity described above.

In [ ]:
# ====================================================================
# Customization templates — one per artifact type. Edit the values you
# care about; defaults are SDK defaults. Then set ARTIFACT_TYPE and
# CUSTOMIZATION at the bottom.
# ====================================================================

# ---- FLASHCARDS ------------------------------------------------------
# numberOfCards: 1 = Fewer, 2 = Standard/More  (API rejects 3; SDK maps 3→2)
# difficulty:    1 = Easy,  2 = Medium, 3 = Hard
# language:      BCP-47 (omit to use notebook default)
FLASHCARDS_CUSTOMIZATION = {
    "numberOfCards": 2,
    "difficulty":    2,
    # "language": "en",
}
FLASHCARDS_INSTRUCTIONS = ""  # e.g. "Focus on key definitions and any formulas."

# ---- QUIZ ------------------------------------------------------------
# numberOfQuestions: 1 = Fewer, 2 = Standard, 3 = More
# difficulty:        1 = Easy,  2 = Medium,   3 = Hard
QUIZ_CUSTOMIZATION = {
    "numberOfQuestions": 2,
    "difficulty":        2,
    # "language": "en",
}
QUIZ_INSTRUCTIONS = ""

# ---- VIDEO -----------------------------------------------------------\
# FORMAT:       1 = Explainer   2 = Brief      3 = Cinematic  (Cinematic ignores visualStyle)
# VISUAL STYLE: 1 = Auto        2 = Classic    3 = Whiteboard   4 = Heritage  9 = Kawaii
# focus:       free-text — what should the AI hosts focus on (any style)
VIDEO_CUSTOMIZATION = {
    "format":      1,
    "visualStyle": 4,
    "focus":       "",
    # "customStyleDescription": "...",   # required only when visualStyle == 2
    # "language": "en",
}
VIDEO_INSTRUCTIONS = VIDEO_PROMPT.strip()  # separate top-level instructions (SDK passes both)

# ---- AUDIO -----------------------------------------------------------
# format: 0 = Deep dive, 1 = Brief, 2 = Critique, 3 = Debate
# length: 1 = Short, 2 = Default, 3 = Long
#   Deep dive (0)  → 1/2/3 valid
#   Brief     (1)  → length is ignored
#   Critique  (2)  → 1/2 only
#   Debate    (3)  → 1/2 only
AUDIO_CUSTOMIZATION = {
    "format": 0,
    "length": 2,
    # "language": "en",
}
AUDIO_INSTRUCTIONS = ""

# ---- SLIDE_DECK ------------------------------------------------------
# format: 2 = Presenter (concise), 3 = Detailed deck
# length: 2 = Short (5–10 slides), 3 = Default (10–15 slides)
SLIDE_DECK_CUSTOMIZATION = {
    "format": 3,
    "length": 3,
    # "language": "en",
}
SLIDE_DECK_INSTRUCTIONS = SLIDES_PROMPT.strip()

# ---- INFOGRAPHIC -----------------------------------------------------
# orientation:   1 = Landscape, 2 = Portrait, 3 = Square
# levelOfDetail: 1 = Concise,   2 = Standard, 3 = Detailed
INFOGRAPHIC_CUSTOMIZATION = {
    "orientation":   3,
    "levelOfDetail": 3,
    # "language": "en",
}
INFOGRAPHIC_INSTRUCTIONS = INFOGRAPHIC_PROMPT.strip()

# ---- REPORT  (no customization supported by the SDK) -----------------
REPORT_CUSTOMIZATION  = None
REPORT_INSTRUCTIONS   = ""

# ---- MIND_MAP  (no customization supported by the SDK) ---------------
MIND_MAP_CUSTOMIZATION = None
MIND_MAP_INSTRUCTIONS  = ""

# ====================================================================
# Pick ONE: change ARTIFACT_TYPE and the two *_CUSTOMIZATION/
# *_INSTRUCTIONS pointers will follow.
# ====================================================================

ARTIFACT_TYPE = "slide_deck"   # FLASHCARDS | QUIZ | VIDEO | AUDIO | SLIDE_DECK | INFOGRAPHIC | REPORT | MIND_MAP

# Verified - Video, Infographic, Slide Deck

_TEMPLATES = {
    "FLASHCARDS":  (FLASHCARDS_CUSTOMIZATION,  FLASHCARDS_INSTRUCTIONS),
    "QUIZ":        (QUIZ_CUSTOMIZATION,        QUIZ_INSTRUCTIONS),
    "VIDEO":       (VIDEO_CUSTOMIZATION,       VIDEO_INSTRUCTIONS),
    "AUDIO":       (AUDIO_CUSTOMIZATION,       AUDIO_INSTRUCTIONS),
    "SLIDE_DECK":  (SLIDE_DECK_CUSTOMIZATION,  SLIDE_DECK_INSTRUCTIONS),
    "INFOGRAPHIC": (INFOGRAPHIC_CUSTOMIZATION, INFOGRAPHIC_INSTRUCTIONS),
    "REPORT":      (REPORT_CUSTOMIZATION,      REPORT_INSTRUCTIONS),
    "MIND_MAP":    (MIND_MAP_CUSTOMIZATION,    MIND_MAP_INSTRUCTIONS),
}
CUSTOMIZATION, INSTRUCTIONS = _TEMPLATES[ARTIFACT_TYPE.upper()]
print(f"Artifact type:  {ARTIFACT_TYPE}")
print(f"Customization:  {CUSTOMIZATION}")
print(f"Instructions:   {INSTRUCTIONS!r}"[:200])

Artifact type:  slide_deck
Customization:  {'format': 3, 'length': 3}
Instructions:   'Make a well structured, practical guide to understanding and applying the techniques in this source.\nUse a moderately dark, very colorful, multi colored, sleek theme. Professional lo


## List Sources

`list_sources(notebook_id, creds)` — prints an indexed table of all sources (title, status, full ID).

In [ ]:
from pipeline import list_sources

sources = list_sources(NOTEBOOK_ID, creds)

# Filter examples:
# sources = sources[:1]                   # first one (good for testing)
# sources = [sources[i] for i in [0, 2]]  # specific picks


Notebook :  4 Napoleon
+---+----------------------+------------+--------------------------------------+
| # | Title                | Status     | Source ID                            |
+---+----------------------+------------+--------------------------------------+
| 0 | 00. Introduction.txt | READY      | 89fd686f-2cd5-4ef0-b427-50424c8d24b4 |
| 1 | 01. Corsica.txt      | READY      | ce3a771d-70b1-44ee-949c-8e83e4a21d52 |
| 2 | 02. Revolution.txt   | READY      | ebd71561-b799-4de4-bde9-1086cef8ae7e |
| 3 | 03. Desire.txt       | READY      | 5f2ccf0c-095a-499a-889f-9abb1bc92525 |
| 4 | 04. Italy.txt        | READY      | 56a7c070-e2d7-41f0-8467-5fe755e38f42 |
| 5 | 05. Victory.txt      | READY      | 36ac048c-22b5-403d-be4e-8d80f1cc68fc |
| 6 | 06. Peace.txt        | READY      | 3677333b-9223-4605-8920-f0b914ab52a5 |
| 7 | 07. Egypt.txt        | READY      | 8606ebfd-a9fb-443e-9779-cda83dac7c94 |
| 8 | 08. Acre.txt         | READY      | 5352bfee-94aa-4f22-bff3-7f9984c43aa6 |
| 9 

In [ ]:
from pipeline import list_artifacts

artifacts = list_artifacts(NOTEBOOK_ID, sources, creds)


Artifacts in notebook 0a79eb08-bbdb-48a0-9b23-5a2ca44e368f (61 total)
+----+----------------------------------------------+-------------+---------------------+---------+--------------------------------------+
| #  | Title                                        | Type        | Created             | Sources | Artifact ID                          |
+----+----------------------------------------------+-------------+---------------------+---------+--------------------------------------+
|  0 | The Napoleonic Playbook                      | SLIDE_DECK  | 2026-05-31 22:11:10 | 0       | 321d77f1-1abc-48bd-9885-11d866f47434 |
|  1 | The Napoleonic Playbook                      | SLIDE_DECK  | 2026-05-31 21:49:15 | 0       | 2d278ef6-3dce-4588-9f9c-b4dcb7924501 |
|  2 | Napoleonic Executive Architecture            | SLIDE_DECK  | 2026-05-31 21:46:38 | 0       | bdb0b73b-ef50-4def-9468-b1d7008f9ba2 |
|  3 | Napoleonic Strategy Decoded                  | SLIDE_DECK  | 2026-05-31 21:47:53 | 0    

## Create Artifacts

**Single entry point:** `create_artifacts(notebook_id, type, sources, customization, instructions, creds, *, dry_run=False)`

- One job per entry in `sources`
- Jobs saved to `jobs/<type>/<timestamp>_<NotebookName>.json`
- `dry_run=True` — preview without calling the API
- Skip this cell and use the **Resume** cell below if jobs were already submitted

In [ ]:
from pipeline import create_artifacts

jobs = create_artifacts(
    NOTEBOOK_ID,
    ARTIFACT_TYPE,
    sources[0],
    CUSTOMIZATION,
    INSTRUCTIONS or None,
    creds,
    dry_run=False,   # ← set to False to actually submit
)


Submitted 1 job(s)  →  jobs\slide_deck\20260531_221112 4 Napoleon.json
+---+----------------------+----------------------------------------------+
| # | Source               | Artifact ID                                  |
+---+----------------------+----------------------------------------------+
| 0 | 00. Introduction.txt | 321d77f1-1abc-48bd-9885-11d866f47434         |
+---+----------------------+----------------------------------------------+


## Poll Until Ready

Use the **resume cell** first if you're continuing a previous session (jobs already submitted).

In [ ]:
# ── Resume cell — run this instead of Create if you already submitted jobs ──
from pipeline import load_jobs
jobs = load_jobs(ARTIFACT_TYPE)

In [ ]:
from pipeline import poll_jobs

# VIDEO and AUDIO take longer — use a larger interval / timeout.
_INTERVALS = {
    "VIDEO": (60, 60),
    "AUDIO": (60, 30),
    "SLIDE_DECK": (30, 20),
}
interval, max_wait = _INTERVALS.get(ARTIFACT_TYPE, (15, 15))

poll_jobs(jobs, NOTEBOOK_ID, creds, interval=interval, max_wait_min=max_wait)

  ✓ 00. Introduction.txt                                    READY

✅ All artifacts ready — proceed to download.


True

## Download

Each artifact is saved into `outputs/<artifact_type>/<NotebookName>/`. File extensions are chosen by the SDK (JSON for flashcards/quiz/infographic, MP3 for audio, MP4 for video, PDF for slide deck).

In [ ]:
from pipeline import download_artifacts

download_artifacts(jobs, NOTEBOOK_ID, ARTIFACT_TYPE, creds)


Downloaded 1 / 1 artifact(s)  →  d:\Core\_Code D\notebooklm-kit\outputs\slide_deck\4 Napoleon
+---+----------------------+----------------------------------------------+------------+
| # | Source               | File                                         | Status     |
+---+----------------------+----------------------------------------------+------------+
| 0 | 00. Introduction.txt | 20260531 221110 00. Introduction             | downloaded |
+---+----------------------+----------------------------------------------+------------+


[{'sourceTitle': '00. Introduction.txt',
  'notebookTitle': ' 4 Napoleon',
  'createdAt': '2026-05-31T16:41:10.142Z',
  'filePath': 'd:\\Core\\_Code D\\notebooklm-kit\\outputs\\slide_deck\\4 Napoleon\\20260531 221110 00. Introduction',
  'status': 'downloaded'}]